In [37]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

filePath = '../../data/processed/combinedWithOdds.csv'
df = pd.read_csv(filePath)

columns_to_keep = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'Attendance', 'Referee', 'HS', 'AS', 'HST', 'AST',
    'HHW', 'AHW', 'HC', 'AC', 'HF', 'AF', 'HFKC', 'AFKC', 'HO', 'AO', 'HY', 'AY',
    'HR', 'AR', '1XBH', '1XBD', '1XBA', 'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA', 'BFDH', 'BFDD', 'BFDA',
    'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'PSH', 'PH', 'PSD', 'PD', 'PSA', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA',
    'SYH', 'SYD', 'SYA', 'WHH', 'WHD', 'WHA',
    'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA',
    'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]
df = df[[col for col in columns_to_keep if col in df.columns]]

drop_cols = ['Time', 'Attendance', 'HHW', 'AHW', 'HO', 'AO', 'Div']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

def date_format_type(date_str):
    if not isinstance(date_str, str):
        return "not_a_string"
    patterns = {
        "%d/%m/%y": r"^\d{2}/\d{2}/\d{2}$",
        "%d/%m/%Y": r"^\d{2}/\d{2}/\d{4}$",
        "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
        "%m-%d-%Y": r"^\d{2}-\d{2}-\d{4}$",
        "%Y/%m/%d": r"^\d{4}/\d{2}/\d{2}$",
    }
    for fmt, pat in patterns.items():
        if re.match(pat, date_str):
            return fmt
    return "unknown"

df['DateFormat'] = df['Date'].apply(date_format_type)

def parse_dates(row):
    date_str = row['Date']
    if isinstance(date_str, str):
        for fmt in ("%d/%m/%y", "%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%Y/%m/%d"):
            try:
                return pd.to_datetime(date_str, format=fmt)
            except Exception:
                continue
        return pd.NaT
    else:
        return pd.NaT

df['Date'] = df.apply(parse_dates, axis=1)
df = df.drop(columns=['DateFormat'], errors='ignore')
df = df[df['Date'] >= pd.Timestamp('2000-08-18')]
df = df.reset_index(drop=True)

def get_season(date):
    if pd.isnull(date):
        return np.nan
    year = date.year
    month = date.month
    if month >= 8: 
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"
df['Season'] = df['Date'].apply(get_season)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

odds_cols = [
     'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'IWH', 'IWD', 'IWA',
    'WHH', 'WHD', 'WHA',
    'PSH', 'PSD', 'PSA', 'PH', 'PD', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    '1XBH', '1XBD', '1XBA',
    'BWH', 'BWD', 'BWA',
    'SOH', 'SOD', 'SOA',
    'SBH', 'SBD', 'SBA',
    'CLH', 'CLD', 'CLA',
    'BMGMH', 'BMGMD', 'BMGMA',
    'BFDH', 'BFDD', 'BFDA',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA',
    'SYH', 'SYD', 'SYA',
    'SJH', 'SJD', 'SJA',
    'BSH', 'BSD', 'BSA',
    'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]

score_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',  
   'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]
for col in odds_cols + score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates()

essential_odds = ['B365H', 'B365D', 'B365A', 'WHH', 'WHD', 'WHA', 'IWH', 'IWD', 'IWA']
core_odds = odds_cols

def clean_odds_dataframe(df, essential_odds, core_odds, fill_method='mean'):
    essentials = [col for col in essential_odds if col in df.columns]
    df = df.dropna(subset=essentials).reset_index(drop=True)
    for col in core_odds:
        if col in df.columns:
            fill_value = df[col].mean() if fill_method == 'mean' else df[col].median() if fill_method == 'median' else 0 if fill_method == 'zero' else -1 if fill_method == 'negone' else None
            if fill_value is not None:
                df[col] = df[col].fillna(fill_value)
    return df

df = clean_odds_dataframe(df, essential_odds, core_odds, fill_method='mean')
df = df.reset_index(drop=True)

if all(col in df.columns for col in ['FTHG', 'FTAG']):
    df['TotalGoals'] = df['FTHG'] + df['FTAG']
    df['GoalsOver2_5'] = (df['TotalGoals'] > 2.5).astype(int)
    df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    df['Home_2plus'] = (df['FTHG'] >= 2).astype(int)
    df['Away_2plus'] = (df['FTAG'] >= 2).astype(int)

df = df.sort_values('Date')
df['HomeTeam_mean_FTHG'] = (
    df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
)
df['AwayTeam_mean_FTAG'] = (
    df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).expanding().mean())
)

df['HomeTeam_str'] = df['HomeTeam']
df['AwayTeam_str'] = df['AwayTeam']
df = pd.get_dummies(df, columns=['HomeTeam', 'AwayTeam'])
df = df.rename(columns={'HomeTeam_str': 'HomeTeam', 'AwayTeam_str': 'AwayTeam'})

def add_recent_form_features(df, n_matches=5):
    base = df.copy().sort_values('Date')
    home_df = base[['Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsFor', 'FTAG': 'GoalsAgainst'})
    away_df = base[['Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsFor', 'FTHG': 'GoalsAgainst'})
    results = pd.concat([home_df, away_df], ignore_index=True).sort_values(['Team', 'Date'])
    def get_points(row):
        return 3 if row['GoalsFor'] > row['GoalsAgainst'] else (1 if row['GoalsFor'] == row['GoalsAgainst'] else 0)
    results['Points'] = results.apply(get_points, axis=1)
    results['RollingGF'] = results.groupby('Team')['GoalsFor'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingGA'] = results.groupby('Team')['GoalsAgainst'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingPoints'] = results.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).sum())
    def get_form(row, team_col):
        team = row[team_col]
        date = row['Date']
        row_form = results[(results['Team'] == team) & (results['Date'] < date)].sort_values('Date').tail(1)
        if row_form.empty:
            return pd.Series([np.nan, np.nan, np.nan])
        return row_form[['RollingGF', 'RollingGA', 'RollingPoints']].values[0]
    base[['HomeRecentGF', 'HomeRecentGA', 'HomeRecentPts']] = base.apply(
        lambda row: get_form(row, 'HomeTeam'), axis=1, result_type='expand')
    base[['AwayRecentGF', 'AwayRecentGA', 'AwayRecentPts']] = base.apply(
        lambda row: get_form(row, 'AwayTeam'), axis=1, result_type='expand')
    return base

df = add_recent_form_features(df, n_matches=5)
df = df.dropna(subset=['HomeRecentGF', 'AwayRecentGF'])

df = df.sort_values('Date')
h2h_home_wins, team_stats, home_positions, away_positions = [], {}, [], []
for idx, row in df.iterrows():
    home, away, match_date = row['HomeTeam'], row['AwayTeam'], row['Date']
    prev_matches = df[
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home))) & (df['Date'] < match_date)
    ].sort_values('Date', ascending=False).head(5)
    home_wins = ((prev_matches['HomeTeam'] == home) & (prev_matches['FTHG'] > prev_matches['FTAG'])).sum()
    h2h_home_wins.append(home_wins)
    league_table = [{'team': t, 'points': s['points'], 'gd': s['gd'], 'scored': s['scored']} for t, s in team_stats.items()]
    table_df = pd.DataFrame(league_table)
    if not table_df.empty:
        table_df = table_df.sort_values(['points', 'gd', 'scored'], ascending=[False, False, False])
        table_df['position'] = range(1, len(table_df) + 1)
        home_pos = table_df[table_df['team'] == home]['position'].values[0] if home in table_df['team'].values else len(table_df) + 1
        away_pos = table_df[table_df['team'] == away]['position'].values[0] if away in table_df['team'].values else len(table_df) + 1
    else:
        home_pos = away_pos = 1
    home_positions.append(home_pos)
    away_positions.append(away_pos)
    home_goals, away_goals = row['FTHG'], row['FTAG']
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {'points': 0, 'gd': 0, 'scored': 0}
    if home_goals > away_goals:
        team_stats[home]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away]['points'] += 3
    else:
        team_stats[home]['points'] += 1
        team_stats[away]['points'] += 1
    team_stats[home]['gd'] += home_goals - away_goals
    team_stats[away]['gd'] += away_goals - home_goals
    team_stats[home]['scored'] += home_goals
    team_stats[away]['scored'] += away_goals

df['h2h_home_wins_last5'] = h2h_home_wins
df['home_league_position'] = home_positions
df['away_league_position'] = away_positions
df['position_diff'] = df['home_league_position'] - df['away_league_position']

N = 5
df['HomePts'] = np.where(df['FTHG'] > df['FTAG'], 3, np.where(df['FTHG'] == df['FTAG'], 1, 0))
df['AwayPts'] = np.where(df['FTAG'] > df['FTHG'], 3, np.where(df['FTAG'] == df['FTHG'], 1, 0))
df['HomeRecentPts'] = (
    df.groupby('HomeTeam')['HomePts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentPts'] = (
    df.groupby('AwayTeam')['AwayPts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['HomeGoalDiff'] = df['FTHG'] - df['FTAG']
df['AwayGoalDiff'] = df['FTAG'] - df['FTHG']
df['HomeRecentGoalDiff'] = (
    df.groupby('HomeTeam')['HomeGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentGoalDiff'] = (
    df.groupby('AwayTeam')['AwayGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentGoalDiff'] = df['HomeRecentGoalDiff'] - df['AwayRecentGoalDiff']
df['HomeRecentShotsOnTarget'] = (
    df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentShotsOnTarget'] = (
    df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentShotsOnTargetDiff'] = df['HomeRecentShotsOnTarget'] - df['AwayRecentShotsOnTarget']
df['PosDiff'] = df['home_league_position'] - df['away_league_position']

for col in ['B365>2.5', 'B365<2.5']:
    median = df[col].median()
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(median)

df['B365>2.5_implied_prob'] = 1 / df['B365>2.5']
df['B365<2.5_implied_prob'] = 1 / df['B365<2.5']
df['OddsMargin'] = df['B365H'] / df['B365A']
df['OU_OddsMargin'] = df['B365>2.5'] / df['B365<2.5']
df['OverUnderRatio'] = df['B365>2.5'] / df['B365<2.5']

df['Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['EarlySeason'] = (df['Month'] <= 3).astype(int)

def rolling_ref_aggression(subdf):
    agg = subdf[['HY', 'AY', 'HR', 'AR']].shift(1).sum(axis=1)
    return agg.rolling(10, min_periods=1).mean()
df = df.sort_values('Date')
df['RefereeAggression'] = (
    df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)
)

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=['number']):
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'category']):
    df[col] = df[col].fillna(df[col].mode()[0])
df = df.drop_duplicates().reset_index(drop=True)

threshold = 0.95 
df = df.loc[:, df.isnull().mean() < threshold]

def safe_agg(df, cols, agg='mean'):
    valid_cols = [c for c in cols if c in df.columns]
    if valid_cols:
        if agg == 'mean':
            return df[valid_cols].mean(axis=1)
        elif agg == 'min':
            return df[valid_cols].min(axis=1)
        elif agg == 'max':
            return df[valid_cols].max(axis=1)
    return np.nan

home_win_cols = ['B365H', 'BSH', 'BWH', 'GBH', 'IWH', 'LBH', 'PSH', 'SOH', 'SBH', 'SJH', 'VCH', 'WHH']
draw_cols = ['B365D', 'BSD', 'BWD', 'GBD', 'IWD', 'LBD', 'PSD', 'SOD', 'SBD', 'SJD', 'VCD', 'WHD']
away_win_cols = ['B365A', 'BSA', 'BWA', 'GBA', 'IWA', 'LBA', 'PSA', 'SOA', 'SBA', 'SJA', 'VCA', 'WHA']

df['cons_mean_home'] = safe_agg(df, home_win_cols, 'mean')
df['cons_mean_draw'] = safe_agg(df, draw_cols, 'mean')
df['cons_mean_away'] = safe_agg(df, away_win_cols, 'mean')
df['cons_min_home'] = safe_agg(df, home_win_cols, 'min')
df['cons_min_draw'] = safe_agg(df, draw_cols, 'min')
df['cons_min_away'] = safe_agg(df, away_win_cols, 'min')
df['cons_max_home'] = safe_agg(df, home_win_cols, 'max')
df['cons_max_draw'] = safe_agg(df, draw_cols, 'max')
df['cons_max_away'] = safe_agg(df, away_win_cols, 'max')

df['home_odds_spread'] = df['cons_max_home'] - df['cons_min_home']
df['draw_odds_spread'] = df['cons_max_draw'] - df['cons_min_draw']
df['away_odds_spread'] = df['cons_max_away'] - df['cons_min_away']

if all(col in df.columns for col in ['B365H', 'B365D', 'B365A']):
    df['B365_overround'] = (1 / df['B365H']) + (1 / df['B365D']) + (1 / df['B365A'])
if all(col in df.columns for col in ['cons_mean_home', 'cons_mean_draw', 'cons_mean_away']):
    df['cons_overround'] = (1 / df['cons_mean_home']) + (1 / df['cons_mean_draw']) + (1 / df['cons_mean_away'])

ou_cols_over = ['BbMx>2.5', 'BbAv>2.5', 'GB>2.5', 'B365>2.5', 'P>2.5', 'Max>2.5', 'Avg>2.5']
ou_cols_under = ['BbMx<2.5', 'BbAv<2.5', 'GB<2.5', 'B365<2.5', 'P<2.5', 'Max<2.5', 'Avg<2.5']
df['cons_mean_over2.5'] = safe_agg(df, ou_cols_over, 'mean')
df['cons_mean_under2.5'] = safe_agg(df, ou_cols_under, 'mean')
df['cons_over2.5_prob'] = 1 / df['cons_mean_over2.5']
df['cons_under2.5_prob'] = 1 / df['cons_mean_under2.5']

ah_home_cols = ['BbMxAHH', 'BbAvAHH', 'GBAHH', 'LBAHH', 'B365AHH', 'PAHH', 'MaxAHH', 'AvgAHH']
ah_away_cols = ['BbMxAHA', 'BbAvAHA', 'GBAHA', 'LBAHA', 'B365AHA', 'PAHA', 'MaxAHA', 'AvgAHA']
df['mean_AH_home'] = safe_agg(df, ah_home_cols, 'mean')
df['mean_AH_away'] = safe_agg(df, ah_away_cols, 'mean')
if 'AHh' in df.columns:
    df['market_AHh'] = df['AHh']

if 'FTHG' in df.columns:
    df['target_home_plus_two'] = (df['FTHG'] >= 2).astype(int)
if 'FTAG' in df.columns:
    df['target_away_plus_two'] = (df['FTAG'] >= 2).astype(int)

df['home_away_odds_ratio'] = df['cons_mean_home'] / df['cons_mean_away']

print("Shape of df:", df.shape)
print("Columns:", df.columns.tolist())

/var/folders/tk/jl3yfcz52s9020z4_8dbfkvw0000gn/T/ipykernel_78006/2589973465.py:273: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)


Shape of df: (7901, 260)
Columns: ['Date', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH', 'VCH', 'VCD', 'VCA', 'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH', 'PSH', 'PSD', 'PSA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA', 'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA', 'WHH', 'WHD', 'WHA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh', 'Season', 'Year', 'Month', 'DayOfWeek', 'TotalGoals', 'GoalsOver2_5', 'BTTS', 'Home_2pl

In [38]:
df.to_csv("../../data/processed/testtest.csv", index=False)

In [39]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression,cons_mean_home,cons_mean_draw,cons_mean_away,cons_min_home,cons_min_draw,cons_min_away,cons_max_home,cons_max_draw,cons_max_away,home_odds_spread,draw_odds_spread,away_odds_spread,B365_overround,cons_overround,cons_mean_over2.5,cons_mean_under2.5,cons_over2.5_prob,cons_under2.5_prob,mean_AH_home,mean_AH_away,market_AHh,target_home_plus_two,target_away_plus_two,home_away_odds_ratio
0,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,1.16,5.50,12.00,41.0,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1